# 03 — Baseline Models

**Goal**: Establish minimum performance bars. Every future model must beat these.  
**Protocol**: Stratified 5-fold CV, report mean ± std for ROC-AUC, F1, Recall.  
**Logging**: All runs tracked in MLflow experiment `Heart-Disease-Kaggle`.


In [1]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import json, pathlib, time

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_curve, auc

from src.data_utils import load_data, get_X_y, FEATURE_COLS, TARGET
from src.evaluation import cv_evaluate, log_mlflow_run
from src.visualization import save_fig, plot_roc_curves, PALETTE

sns.set_theme(style='whitegrid', palette=PALETTE)
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('Heart-Disease-Kaggle')

train = load_data('train')
X, y = get_X_y(train, extra_features=False)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

RESULTS_DIR = pathlib.Path('../results/metrics')
all_results = []
print(f'X shape: {X.shape}, class balance: {y.mean():.3f}')

X shape: (630000, 13), class balance: 0.448


In [2]:
def run_model(name, model, X, y, cv, phase='baseline', feature_set='baseline', params=None):
    """Run CV, log to MLflow, return result dict."""
    t0 = time.time()
    metrics = cv_evaluate(model, X, y, cv=cv)
    elapsed = time.time() - t0
    
    run_id = log_mlflow_run(
        model_name=name,
        metrics=metrics,
        params=params or {},
        tags={'phase': phase, 'feature_set': feature_set},
    )
    
    result = {'model': name, 'phase': phase, **metrics, 'elapsed_s': round(elapsed, 1)}
    print(f'  {name:<40} AUC={metrics["roc_auc_mean"]:.4f}±{metrics["roc_auc_std"]:.4f}  '
          f'F1={metrics["f1_mean"]:.4f}  Recall={metrics["recall_mean"]:.4f}  '
          f'[{elapsed:.0f}s]')
    return result

## 3.1 Dummy Classifiers (Floor)

In [3]:
print('--- Dummy Classifiers ---')
dummies = [
    ('Dummy (majority)',   DummyClassifier(strategy='most_frequent', random_state=42)),
    ('Dummy (stratified)', DummyClassifier(strategy='stratified', random_state=42)),
    ('Dummy (prior)',      DummyClassifier(strategy='prior', random_state=42)),
]
for name, model in dummies:
    r = run_model(name, model, X, y, CV, phase='baseline')
    all_results.append(r)

--- Dummy Classifiers ---


/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control

  Dummy (majority)                         AUC=0.5000±0.0000  F1=0.0000  Recall=0.0000  [2s]


  Dummy (stratified)                       AUC=0.4990±0.0010  F1=0.4471  Recall=0.4470  [1s]


/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control

  Dummy (prior)                            AUC=0.5000±0.0000  F1=0.0000  Recall=0.0000  [1s]


/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 3.2 Logistic Regression Variants

In [4]:
print('--- Logistic Regression Variants ---')
ss = StandardScaler()
X_scaled = pd.DataFrame(ss.fit_transform(X), columns=X.columns, index=X.index)

lr_models = [
    ('LR (L2, C=1.0)',    LogisticRegression(penalty='l2', C=1.0,   max_iter=1000, random_state=42),
     {'penalty': 'l2', 'C': 1.0}),
    ('LR (L1, C=1.0)',    LogisticRegression(penalty='l1', C=1.0,   max_iter=1000, random_state=42, solver='liblinear'),
     {'penalty': 'l1', 'C': 1.0}),
    ('LR (ElasticNet)',   LogisticRegression(penalty='elasticnet', C=1.0, max_iter=1000, random_state=42, solver='saga', l1_ratio=0.5),
     {'penalty': 'elasticnet', 'l1_ratio': 0.5}),
    ('LR (L2, C=0.01)',   LogisticRegression(penalty='l2', C=0.01,  max_iter=1000, random_state=42),
     {'penalty': 'l2', 'C': 0.01}),
    ('LR (L2, C=100)',    LogisticRegression(penalty='l2', C=100,   max_iter=1000, random_state=42),
     {'penalty': 'l2', 'C': 100}),
]
for name, model, params in lr_models:
    r = run_model(name, model, X_scaled, y, CV, phase='baseline', feature_set='scaled', params=params)
    all_results.append(r)

--- Logistic Regression Variants ---


  LR (L2, C=1.0)                           AUC=0.9505±0.0003  F1=0.8679  Recall=0.8563  [1s]


  LR (L1, C=1.0)                           AUC=0.9505±0.0003  F1=0.8679  Recall=0.8564  [2s]


  LR (ElasticNet)                          AUC=0.9505±0.0003  F1=0.8679  Recall=0.8564  [4s]


  LR (L2, C=0.01)                          AUC=0.9505±0.0003  F1=0.8679  Recall=0.8562  [1s]


  LR (L2, C=100)                           AUC=0.9505±0.0003  F1=0.8679  Recall=0.8563  [2s]


## 3.3 Gaussian Naive Bayes

In [5]:
print('--- Naive Bayes ---')
gnb_models = [
    ('GaussianNB',     GaussianNB(), {}),
    ('GaussianNB (var_smoothing=1e-5)', GaussianNB(var_smoothing=1e-5), {'var_smoothing': 1e-5}),
]
for name, model, params in gnb_models:
    r = run_model(name, model, X_scaled, y, CV, phase='baseline', feature_set='scaled', params=params)
    all_results.append(r)

--- Naive Bayes ---


  GaussianNB                               AUC=0.9382±0.0005  F1=0.8560  Recall=0.8557  [1s]


  GaussianNB (var_smoothing=1e-5)          AUC=0.9382±0.0005  F1=0.8560  Recall=0.8557  [0s]


## 3.4 K-Nearest Neighbors

In [6]:
print('--- K-Nearest Neighbors ---')
# Use a sample for KNN on 630K (very slow otherwise)
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=30000, replace=False)
X_knn = X_scaled.iloc[sample_idx].reset_index(drop=True)
y_knn = y.iloc[sample_idx].reset_index(drop=True)

cv_knn = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn_models = [
    ('KNN (k=3)',  KNeighborsClassifier(n_neighbors=3,  n_jobs=-1), {'k': 3}),
    ('KNN (k=5)',  KNeighborsClassifier(n_neighbors=5,  n_jobs=-1), {'k': 5}),
    ('KNN (k=11)', KNeighborsClassifier(n_neighbors=11, n_jobs=-1), {'k': 11}),
    ('KNN (k=21)', KNeighborsClassifier(n_neighbors=21, n_jobs=-1), {'k': 21}),
]
for name, model, params in knn_models:
    r = run_model(name, model, X_knn, y_knn, cv_knn, phase='baseline', feature_set='scaled_30k', params=params)
    r['note'] = 'sample=30K'
    all_results.append(r)

--- K-Nearest Neighbors ---


  KNN (k=3)                                AUC=0.9018±0.0043  F1=0.8386  Recall=0.8305  [1s]


  KNN (k=5)                                AUC=0.9209±0.0041  F1=0.8461  Recall=0.8383  [1s]


  KNN (k=11)                               AUC=0.9354±0.0025  F1=0.8544  Recall=0.8427  [2s]


  KNN (k=21)                               AUC=0.9403±0.0021  F1=0.8566  Recall=0.8431  [2s]


## 3.5 Decision Trees

In [7]:
print('--- Decision Trees ---')
dt_models = [
    ('Decision Tree (depth=3)',  DecisionTreeClassifier(max_depth=3,  random_state=42), {'max_depth': 3}),
    ('Decision Tree (depth=5)',  DecisionTreeClassifier(max_depth=5,  random_state=42), {'max_depth': 5}),
    ('Decision Tree (depth=10)', DecisionTreeClassifier(max_depth=10, random_state=42), {'max_depth': 10}),
    ('Decision Tree (unlimited)',DecisionTreeClassifier(max_depth=None, random_state=42), {'max_depth': 'None'}),
]
for name, model, params in dt_models:
    r = run_model(name, model, X, y, CV, phase='baseline', feature_set='baseline', params=params)
    all_results.append(r)

--- Decision Trees ---


  Decision Tree (depth=3)                  AUC=0.9058±0.0005  F1=0.8119  Recall=0.7624  [1s]


  Decision Tree (depth=5)                  AUC=0.9327±0.0006  F1=0.8436  Recall=0.8262  [1s]


  Decision Tree (depth=10)                 AUC=0.9488±0.0003  F1=0.8661  Recall=0.8571  [2s]


  Decision Tree (unlimited)                AUC=0.8239±0.0009  F1=0.8059  Recall=0.8077  [2s]


## 3.6 Results Summary

In [8]:
results_df = pd.DataFrame(all_results)
display_cols = ['model', 'roc_auc_mean', 'roc_auc_std', 'f1_mean', 'recall_mean', 'precision_mean']
summary = results_df[display_cols].sort_values('roc_auc_mean', ascending=False)
summary.columns = ['Model', 'ROC-AUC', '±std', 'F1', 'Recall', 'Precision']
print('\n=== BASELINE MODELS LEADERBOARD ===')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


=== BASELINE MODELS LEADERBOARD ===
                          Model  ROC-AUC   ±std     F1  Recall  Precision
                LR (L2, C=0.01)   0.9505 0.0003 0.8679  0.8562     0.8798
                 LR (L1, C=1.0)   0.9505 0.0003 0.8679  0.8564     0.8798
                LR (ElasticNet)   0.9505 0.0003 0.8679  0.8564     0.8798
                 LR (L2, C=1.0)   0.9505 0.0003 0.8679  0.8563     0.8798
                 LR (L2, C=100)   0.9505 0.0003 0.8679  0.8563     0.8798
       Decision Tree (depth=10)   0.9488 0.0003 0.8661  0.8571     0.8753
                     KNN (k=21)   0.9403 0.0021 0.8566  0.8431     0.8706
GaussianNB (var_smoothing=1e-5)   0.9382 0.0005 0.8560  0.8557     0.8563
                     GaussianNB   0.9382 0.0005 0.8560  0.8557     0.8563
                     KNN (k=11)   0.9354 0.0025 0.8544  0.8427     0.8664
        Decision Tree (depth=5)   0.9327 0.0006 0.8436  0.8262     0.8618
                      KNN (k=5)   0.9209 0.0041 0.8461  0.8383     0.8541
 

In [9]:
# Bar chart comparison
non_dummy = results_df[~results_df['model'].str.startswith('Dummy')].copy()
non_dummy = non_dummy.sort_values('roc_auc_mean', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
metrics_plot = [
    ('roc_auc_mean', 'roc_auc_std', 'ROC-AUC'),
    ('f1_mean', 'f1_std', 'F1-Score'),
    ('recall_mean', 'recall_std', 'Recall'),
]
for ax, (mean_col, std_col, title) in zip(axes, metrics_plot):
    colors = sns.color_palette(PALETTE, n_colors=len(non_dummy))
    bars = ax.barh(non_dummy['model'], non_dummy[mean_col], 
                   xerr=non_dummy[std_col], color=colors,
                   capsize=3, error_kw={'elinewidth': 1})
    ax.set_xlabel(title)
    ax.set_title(f'{title} (5-fold CV)')
    ax.set_xlim(max(0, non_dummy[mean_col].min() - 0.05), 1.0)
    for bar, val in zip(bars, non_dummy[mean_col]):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=7)

fig.suptitle('Baseline Models — Cross-Validation Performance', fontsize=13)
fig.tight_layout()
save_fig('03_baseline_results', fig)
plt.show()

In [10]:
# Save results
results_df.to_csv(RESULTS_DIR / '03_baseline_results.csv', index=False)
print(f'Results saved. Best baseline: {summary.iloc[0]["Model"]} AUC={summary.iloc[0]["ROC-AUC"]:.4f}')

Results saved. Best baseline: LR (L2, C=0.01) AUC=0.9505
